In [1]:
%load_ext autoreload
%autoreload 2

In [16]:
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

_ = load_dotenv(find_dotenv())

llm = ChatOpenAI(
    model="gpt-5.6-luna",
    use_responses_api=True, #known bug in LangChain for Luna.
    reasoning={"effort": "low"},  #The reasoning is medium by default so set this to l
)

In [17]:
from deepagents.backends import FilesystemBackend
from deepagents.middleware.filesystem import FilesystemMiddleware
from deepagents.middleware.skills import SkillsMiddleware
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """
    Get the weather for a given location
    """
    return f"The weather in {location} is sunny"

@tool
def get_exchange_rate(currency_from: str, currency_to: str) -> str:
    """
    Get the exchange rate between two currencies
    """
    return f"The exchange rate for {currency_from} to {currency_to} is 1.00"


backend = FilesystemBackend(
    root_dir="../",
    virtual_mode=True,
)

agent = create_agent(
    model=llm,
    tools=[get_weather, get_exchange_rate],
    middleware=[
        # Skill discovery + prompt injection
        SkillsMiddleware(
            backend=backend,
            sources=["./skills/"],
        ),

        # Give the model ONLY read_file
        FilesystemMiddleware(
            backend=backend,
            tools=["read_file"],
            system_prompt=None,
        ),
    ],
)

In [18]:
from langchain.messages import HumanMessage

response = agent.invoke({"messages":[HumanMessage(content="What is the weather in Singapore?")]})

In [19]:
from IPython.display import display, Markdown

display(Markdown(response['messages'][-1].content[0]['text']))

Singapore is currently **sunny**. Enjoy the sunshine—light, breathable clothing should be comfortable.